# 1. Import All Package

In [1]:
# Import experiment dependencies and start the runtime timer.
import os
import time
from datetime import datetime, timezone
import numpy as np
import pandas as pd

from util.seed_config import configure_reproducibility
from util.feedback import to_B
from util.signal_utils import log_signal_density

from model.baseline.standard import WMF
from model.baseline.cofactor import CoFactor, build_item_sppmi_matrix as build_cofactor_item_sppmi_matrix
from model.baseline.rme import RME
from model.baseline.neumf import NeuMF
from model.baseline.lightgcn import LightGCN
from model.proposed.cparms_all import CPARMS_LD, Generator_CPARMS_Liked as _GL, Generator_CPARMS_Disliked as _GD, _net_signal
from model.baseline.itempop import ItemPop

from experiments.global_temporal_split import get_temporal_split
from experiments.hyperparams_set import generate_hyperparam_samples
from experiments.all_ranker import (
    build_user_activity_groups,
    ranking_metrics_at_k,
)
from experiments.significance import significance_table

start_time = time.time()

# 2. Set Up Environment

In [ ]:
# Define the reproducible experiment and evaluation configuration.
TUNING_SEED = 42
SENSITIVITY_SEEDS = (42, 43, 44, 45, 46)
STAT_TEST_SEED = TUNING_SEED

N_ITER_RANDOM_SEARCH = 30
METRIC_KS = (10, 20, 50, 100, 200) 
USER_ACTIVITY_GROUPS = ("interaction_0", "interaction_1", "interaction_2", "interaction_3_plus")

SELECTION_METRIC = "ndcg"
SELECTION_K = 10

MODEL_PARAM_KEY = {
    # Non-personalized baseline
    "01 ItemPop": None,
    # Models sharing the WMF/ALS backbone
    "02 Standard-WMF": "standard_wmf",
    "03 CoFactor": "cofactor_wmf",
    "04 RME": "rme",
    # Models with different backbones
    "05 NeuMF": "neumf",
    "06 LightGCN": "lightgcn",
    # Proposed method
    "07 CPARMS-LD": "cparms_ld",
}

SIGNIFICANCE_PRIMARY_MODEL = "07 CPARMS-LD"

if not SENSITIVITY_SEEDS:
    raise ValueError("Select at least one sensitivity seed.")
if len(set(SENSITIVITY_SEEDS)) != len(SENSITIVITY_SEEDS):
    raise ValueError("SENSITIVITY_SEEDS must contain unique values.")
if STAT_TEST_SEED not in SENSITIVITY_SEEDS:
    raise ValueError("STAT_TEST_SEED must be included in SENSITIVITY_SEEDS.")

RESULTS_DIR = './results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# 3. Load Data

In [ ]:
# Load each selected dataset and create leakage-safe temporal splits.
SELECTED_DATASETS = {
    '01_amz_beauty': './database/csv/dataset_amazon_lux_beauty_5_core.csv',
    '02_amz_industry': './database/csv/dataset_amazon_industry_5_core.csv',
    '03_amz_pantry': './database/csv/dataset_amazon_pantry_5_core.csv',
    '04_amz_music': './database/csv/dataset_amazon_music_5_core.csv',
    '05_amz_instruments': './database/csv/dataset_amazon_instruments_5_core.csv',
}

dataset_configs = []
dataset_eda_all = {}
for dataset_name, dataset_path in SELECTED_DATASETS.items():
    dataset_df = pd.read_csv(dataset_path)
    train_mat, val_mat, train_val_mat, test_mat, eda = get_temporal_split(
        dataset_df,
    )

    dataset_configs.append({
        'name': dataset_name,
        'train': train_mat,
        'val': val_mat,
        'train_val': train_val_mat,
        'test': test_mat,
    })
    dataset_eda_all[dataset_name] = eda

if not dataset_configs:
    raise ValueError('Select at least one dataset.')

# 4. Tune Hyperparameters (One Seed)

In [4]:
# Select model hyperparameters once using the dedicated tuning seed.
best_params_by_tuning_seed = {}

for tuning_seed in (TUNING_SEED,):
    configure_reproducibility(tuning_seed)
    print(f"\n#################### TUNING SEED: {tuning_seed} ####################")

    shared_samples = generate_hyperparam_samples(
        rounds=N_ITER_RANDOM_SEARCH,
        global_seed=tuning_seed,
    )

    seed_best = {}
    best_params_by_tuning_seed[int(tuning_seed)] = seed_best

    for dataset_cfg in dataset_configs:
        dataset_name = dataset_cfg["name"]
        train_mat = dataset_cfg["train"]
        val_mat = dataset_cfg["val"]
        user_count, item_count = train_mat.shape
        print(f"\n========== DATASET: {dataset_name} ==========")

        train_B = to_B(train_mat)
        val_user_groups = build_user_activity_groups(train_B)

        for model_name, param_key in MODEL_PARAM_KEY.items():
            if param_key is None:
                continue
            seed_best[(model_name, dataset_name)] = {
                "best_score": float("-inf"), "best_params": None,
            }
        for round_idx, sample in enumerate(shared_samples):
            print(f"\n--- Round {round_idx + 1}/{N_ITER_RANDOM_SEARCH} ---")
            for model_name, param_key in MODEL_PARAM_KEY.items():
                if param_key is None:
                    continue
                params = sample[param_key]
                print(f"\nTraining {model_name}: {params}")

                if param_key == "standard_wmf":
                    model = WMF(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_mat, n_sweeps=params["n_sweeps"])
                    pred_source = model

                elif param_key == "neumf":
                    model = NeuMF(
                        user_count=user_count,
                        item_count=item_count,
                        latent=params["latent"],
                        hidden_layers=params["hidden_layers"],
                        learning_rate=params["learning_rate"],
                        reg_mf=params["reg_mf"],
                        reg_layers=params["reg_layers"],
                        negative_samples=params["negative_samples"],
                        random_state=params["random_state"],
                    )
                    model.fit(
                        Y=train_mat,
                        epochs=params["epochs"],
                        batch_size=params["batch_size"],
                    )
                    pred_source = model

                elif param_key == "lightgcn":
                    model = LightGCN(
                        user_count=user_count,
                        item_count=item_count,
                        latent=params["latent"],
                        n_layers=params["n_layers"],
                        learning_rate=params["learning_rate"],
                        lambda_rate=params["lambda_rate"],
                        negative_samples=params["negative_samples"],
                        random_state=params["random_state"],
                    )
                    model.fit(
                        Y=train_mat,
                        epochs=params["epochs"],
                        batch_size=params["batch_size"],
                    )
                    pred_source = model

                elif param_key == "cofactor_wmf":
                    model = CoFactor(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        gamma=params["gamma"],
                        lambda_context_rate=params["lambda_context_rate"],
                        negative_samples=params["negative_samples"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_mat, n_sweeps=params["n_sweeps"])
                    pred_source = model

                elif param_key == "rme":
                    model = RME(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        gamma_item_pos=params["gamma_item_pos"],
                        gamma_item_neg=params["gamma_item_neg"],
                        gamma_user_pos=params["gamma_user_pos"],
                        lambda_context_rate=params["lambda_context_rate"],
                        negative_samples=params["negative_samples"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_mat, n_sweeps=params["n_sweeps"])
                    pred_source = model

                elif param_key == "cparms_ld":
                    ld_signal_liked = _GL(
                        k_user=params["k_user"],
                        K_item=params["K_item"],
                        min_support=params["min_support"],
                        min_confidence=params["min_confidence"],
                        min_lift=params["min_lift"],
                        normalize=params["normalize"],
                        random_state=params["random_state"],
                    ).fit_transform(train_mat)
                    log_signal_density("S_liked", ld_signal_liked, user_count, item_count)
                    ld_signal_disliked = _GD(
                        k_user=params["k_user"],
                        K_item=params["K_item"],
                        min_support=params["min_support"],
                        min_confidence=params["min_confidence"],
                        min_lift=params["min_lift"],
                        normalize=params["normalize"],
                        random_state=params["random_state"],
                    ).fit_transform(train_mat)
                    log_signal_density("S_disliked", ld_signal_disliked, user_count, item_count)
                    log_signal_density("S_net (LD)", _net_signal(ld_signal_liked, ld_signal_disliked), user_count, item_count)
                    model = CPARMS_LD(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        gamma_like=params["gamma_like"],
                        gamma_dislike=params["gamma_dislike"],
                        random_state=params["random_state"],
                    )
                    model.fit(
                        Y=train_mat,
                        S_liked=ld_signal_liked,
                        S_disliked=ld_signal_disliked,
                        n_sweeps=params["n_sweeps"],
                        fit_user_mask=(train_mat.getnnz(axis=1) > 0),
                    )
                    pred_source = model

                else:
                    raise ValueError(f"Unsupported model parameter key: {param_key}")

                results_ranking = ranking_metrics_at_k(
                    pred_source=pred_source,
                    train_mat=train_mat,
                    test_mat=val_mat,
                    ks=METRIC_KS,
                    user_groups=val_user_groups,
                )
                all_ndcg = results_ranking["all"]["ndcg"]
                ug = results_ranking["user"]
                print(
                    f"Results: NDCG@10 all={all_ndcg[10]:.6f}, "
                    f"i0={ug['interaction_0']['ndcg'][10]:.6f}, "
                    f"i1={ug['interaction_1']['ndcg'][10]:.6f}, "
                    f"i2={ug['interaction_2']['ndcg'][10]:.6f}, "
                    f"i3+={ug['interaction_3_plus']['ndcg'][10]:.6f}"
                )

                score = results_ranking["all"][SELECTION_METRIC][SELECTION_K]
                tracker = seed_best[(model_name, dataset_name)]
                if score > tracker["best_score"]:
                    tracker["best_score"] = float(score)
                    tracker["best_params"] = dict(params)

                del results_ranking


#################### TUNING SEED: 42 ####################

========== DATASET: 01_amz_beauty ==========

--- Round 1/5 ---

Training 02 Standard-WMF: {'latent': 80, 'lambda_rate': 1.0, 'n_sweeps': 5, 'alpha': 40.0, 'random_state': 42}
[Sweep 1/5] WMF_LOSS: 58217.894127 L2_SUM: 68235.726562 REG: 68235.726562 TOTAL: 126453.620690
[Sweep 5/5] WMF_LOSS: 39455.608538 L2_SUM: 25546.507812 REG: 25546.507812 TOTAL: 65002.116351
Results: NDCG@10 all=0.040947, i0=0.032811, i1=0.048859, i2=0.061766, i3+=0.037937

Training 03 CoFactor: {'latent': 80, 'lambda_rate': 1.0, 'n_sweeps': 25, 'alpha': 40.0, 'lambda_context_rate': 1e-05, 'negative_samples': 1, 'gamma': 0.1, 'random_state': 42}
[Sweep 1/25] WMF_LOSS: 58217.955941 COFACTOR_LOSS: 191.427936 L2_SUM: 68235.726562 CONTEXT_L2_SUM: 8060153.500000 REG: 68316.328097 TOTAL: 126725.711975
[Sweep 5/25] WMF_LOSS: 40739.239061 COFACTOR_LOSS: 7.902678 L2_SUM: 26627.558594 CONTEXT_L2_SUM: 3108603.000000 REG: 26658.644624 TOTAL: 67405.786362
[Sweep 10/25]

# 5. Fixed-Hyperparameter Seed Sensitivity

In [5]:
# Retrain the fixed selected configurations across sensitivity seeds.
final_test_rows = []
significance_rows = []
for experiment_seed in SENSITIVITY_SEEDS:
    configure_reproducibility(experiment_seed)
    print(f"\n#################### [TEST] SEED: {experiment_seed} ####################")
    seed_best = best_params_by_tuning_seed[int(TUNING_SEED)]
    run_significance = int(experiment_seed) == int(STAT_TEST_SEED)

    for dataset_cfg in dataset_configs:
        dataset_name = dataset_cfg["name"]
        train_val_mat = dataset_cfg["train_val"]
        test_mat = dataset_cfg["test"]
        user_count, item_count = train_val_mat.shape
        print(f"\n========== [TEST] DATASET: {dataset_name} ==========")

        train_val_B = to_B(train_val_mat)
        test_user_groups = build_user_activity_groups(train_val_B)
        group_bool_masks = {}
        for group_name, user_idx in test_user_groups.items():
            mask = np.zeros(user_count, dtype=bool)
            mask[user_idx] = True
            group_bool_masks[group_name] = mask

        per_user_by_model = {}
        for model_name, param_key in MODEL_PARAM_KEY.items():
            print(f"\n----- [TEST] MODEL: {model_name} -----")
            s_mat_runtime = 0.0
            log_params = {}
            validation_selection_score = float("nan")

            if param_key is None:
                t0 = time.perf_counter()
                model = ItemPop(user_count=user_count, item_count=item_count)
                model.fit(Y=train_val_mat)
                model_runtime = (time.perf_counter() - t0) / 60.0
                pred_source = model

            else:
                tracker = seed_best[(model_name, dataset_name)]
                selected_params = tracker["best_params"]
                if selected_params is None:
                    print(f"[Skip] No valid best parameters for {model_name}")
                    continue
                params = dict(selected_params)
                params["random_state"] = int(experiment_seed)
                print(f"{model_name} hyperparams: {params}")
                validation_selection_score = tracker["best_score"]

                if param_key == "standard_wmf":
                    t0 = time.perf_counter()
                    model = WMF(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_val_mat, n_sweeps=params["n_sweeps"])
                    model_runtime = (time.perf_counter() - t0) / 60.0
                    pred_source = model
                    log_params = dict(params)

                elif param_key == "neumf":
                    t0 = time.perf_counter()
                    model = NeuMF(
                        user_count=user_count,
                        item_count=item_count,
                        latent=params["latent"],
                        hidden_layers=params["hidden_layers"],
                        learning_rate=params["learning_rate"],
                        reg_mf=params["reg_mf"],
                        reg_layers=params["reg_layers"],
                        negative_samples=params["negative_samples"],
                        random_state=params["random_state"],
                    )
                    model.fit(
                        Y=train_val_mat,
                        epochs=params["epochs"],
                        batch_size=params["batch_size"],
                    )
                    model_runtime = (time.perf_counter() - t0) / 60.0
                    pred_source = model
                    log_params = dict(params)

                elif param_key == "lightgcn":
                    t0 = time.perf_counter()
                    model = LightGCN(
                        user_count=user_count,
                        item_count=item_count,
                        latent=params["latent"],
                        n_layers=params["n_layers"],
                        learning_rate=params["learning_rate"],
                        lambda_rate=params["lambda_rate"],
                        negative_samples=params["negative_samples"],
                        random_state=params["random_state"],
                    )
                    model.fit(
                        Y=train_val_mat,
                        epochs=params["epochs"],
                        batch_size=params["batch_size"],
                    )
                    model_runtime = (time.perf_counter() - t0) / 60.0
                    pred_source = model
                    log_params = dict(params)

                elif param_key == "cofactor_wmf":
                    if params["gamma"] > 0.0:
                        t0 = time.perf_counter()
                        sppmi_mat = build_cofactor_item_sppmi_matrix(
                            train_val_mat,
                            negative_samples=params["negative_samples"],
                        )
                        s_mat_runtime = (time.perf_counter() - t0) / 60.0
                    else:
                        sppmi_mat = None
                    t0 = time.perf_counter()
                    model = CoFactor(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        gamma=params["gamma"],
                        lambda_context_rate=params["lambda_context_rate"],
                        negative_samples=params["negative_samples"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_val_mat, M=sppmi_mat, n_sweeps=params["n_sweeps"])
                    model_runtime = (time.perf_counter() - t0) / 60.0
                    pred_source = model
                    log_params = dict(params)

                elif param_key == "rme":
                    t0 = time.perf_counter()
                    model = RME(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        gamma_item_pos=params["gamma_item_pos"],
                        gamma_item_neg=params["gamma_item_neg"],
                        gamma_user_pos=params["gamma_user_pos"],
                        lambda_context_rate=params["lambda_context_rate"],
                        negative_samples=params["negative_samples"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_val_mat, n_sweeps=params["n_sweeps"])
                    model_runtime = (time.perf_counter() - t0) / 60.0
                    pred_source = model
                    log_params = dict(params)

                elif param_key == "cparms_ld":
                    t0 = time.perf_counter()
                    signal_liked = _GL(
                        k_user=params["k_user"],
                        K_item=params["K_item"],
                        min_support=params["min_support"],
                        min_confidence=params["min_confidence"],
                        min_lift=params["min_lift"],
                        normalize=params["normalize"],
                        random_state=params["random_state"],
                    ).fit_transform(train_val_mat)
                    log_signal_density("S_liked", signal_liked, user_count, item_count)
                    signal_disliked = _GD(
                        k_user=params["k_user"],
                        K_item=params["K_item"],
                        min_support=params["min_support"],
                        min_confidence=params["min_confidence"],
                        min_lift=params["min_lift"],
                        normalize=params["normalize"],
                        random_state=params["random_state"],
                    ).fit_transform(train_val_mat)
                    log_signal_density("S_disliked", signal_disliked, user_count, item_count)
                    log_signal_density("S_net (LD)", _net_signal(signal_liked, signal_disliked), user_count, item_count)
                    s_mat_runtime = (time.perf_counter() - t0) / 60.0
                    t0 = time.perf_counter()
                    model = CPARMS_LD(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        gamma_like=params["gamma_like"],
                        gamma_dislike=params["gamma_dislike"],
                        random_state=params["random_state"],
                    )
                    model.fit(
                        Y=train_val_mat,
                        S_liked=signal_liked,
                        S_disliked=signal_disliked,
                        n_sweeps=params["n_sweeps"],
                        fit_user_mask=(train_val_mat.getnnz(axis=1) > 0),
                    )
                    model_runtime = (time.perf_counter() - t0) / 60.0
                    pred_source = model
                    log_params = dict(params)

                else:
                    raise ValueError(f"Unsupported model parameter key: {param_key}")
            results_ranking = ranking_metrics_at_k(
                pred_source=pred_source,
                train_mat=train_val_mat,
                test_mat=test_mat,
                ks=METRIC_KS,
                user_groups=test_user_groups,
                return_per_user=run_significance,
            )
            if run_significance:
                per_user_by_model[model_name] = results_ranking["per_user"]

            all_ndcg = results_ranking["all"]["ndcg"]
            ug = results_ranking["user"]
            print(
                f"Results: NDCG@10 all={all_ndcg[10]:.6f}, "
                f"i0={ug['interaction_0']['ndcg'][10]:.6f}, "
                f"i1={ug['interaction_1']['ndcg'][10]:.6f}, "
                f"i2={ug['interaction_2']['ndcg'][10]:.6f}, "
                f"i3+={ug['interaction_3_plus']['ndcg'][10]:.6f}"
            )

            row = {
                "seed": int(experiment_seed),
                "dataset_name": dataset_name,
                "model": model_name,
                "s_mat_runtime": float(s_mat_runtime),
                "model_runtime": float(model_runtime),
                "total_runtime": float(s_mat_runtime) + float(model_runtime),
                "n_iter_random_search": N_ITER_RANDOM_SEARCH,
                "selection_metric": SELECTION_METRIC,
                "selection_k": SELECTION_K,
                "selection_group": "all",
                "validation_selection_score": validation_selection_score,
                "n_users_eval": int(results_ranking["n_users_eval"]),
                "n_positive_targets": int(results_ranking["n_positive_targets"]),
            }
            row.update(log_params)
            for k, v in results_ranking["all"]["ndcg"].items():
                row[f"ndcg_all@{int(k)}"] = float(v)
            for g in USER_ACTIVITY_GROUPS:
                for k, v in results_ranking["user"][g]["ndcg"].items():
                    row[f"ndcg_user_{g}@{int(k)}"] = float(v)
            final_test_rows.append(row)

            del results_ranking

        if run_significance and SIGNIFICANCE_PRIMARY_MODEL in per_user_by_model:
            baselines_per_user = {
                name: per_user
                for name, per_user in per_user_by_model.items()
                if name != SIGNIFICANCE_PRIMARY_MODEL
            }
            sig_rows = significance_table(
                primary_per_user=per_user_by_model[SIGNIFICANCE_PRIMARY_MODEL],
                baselines_per_user=baselines_per_user,
                k=SELECTION_K,
                group_masks=group_bool_masks,
            )
            for sig_row in sig_rows:
                sig_row["seed"] = int(experiment_seed)
                sig_row["dataset_name"] = dataset_name
                sig_row["primary_model"] = SIGNIFICANCE_PRIMARY_MODEL
            significance_rows.extend(sig_rows)


#################### [TEST] SEED: 42 ####################

========== [TEST] DATASET: 01_amz_beauty ==========

----- [TEST] MODEL: 01 ItemPop -----
Results: NDCG@10 all=0.017333, i0=0.105978, i1=0.015152, i2=0.000000, i3+=0.000612

----- [TEST] MODEL: 02 Standard-WMF -----
02 Standard-WMF hyperparams: {'latent': 100, 'lambda_rate': 1.0, 'n_sweeps': 20, 'alpha': 5.0, 'random_state': 42}
[Sweep 1/20] WMF_LOSS: 28493.967432 L2_SUM: 3756.699707 REG: 3756.699707 TOTAL: 32250.667139
[Sweep 5/20] WMF_LOSS: 21409.958410 L2_SUM: 3021.026855 REG: 3021.026855 TOTAL: 24430.985265
[Sweep 10/20] WMF_LOSS: 20927.216838 L2_SUM: 2800.443848 REG: 2800.443848 TOTAL: 23727.660686
[Sweep 15/20] WMF_LOSS: 20814.014066 L2_SUM: 2704.677734 REG: 2704.677734 TOTAL: 23518.691801
[Sweep 20/20] WMF_LOSS: 20770.101830 L2_SUM: 2651.877441 REG: 2651.877441 TOTAL: 23421.979271
Results: NDCG@10 all=0.040186, i0=0.037634, i1=0.062699, i2=0.045461, i3+=0.037869

----- [TEST] MODEL: 03 CoFactor -----
03 CoFactor hyperpa

# 6. Review Results and Seed Sensitivity

In [6]:
# Present per-seed results, aggregate sensitivity, and frozen hyperparameters.
df_best_results = pd.DataFrame(final_test_rows)
df_best_results.sort_values(by=["dataset_name", "model", "seed"], inplace=True)

ordered_ndcg_cols = []
for k in METRIC_KS:
    ordered_ndcg_cols.append(f"ndcg_all@{int(k)}")
    ordered_ndcg_cols.extend(
        f"ndcg_user_{g}@{int(k)}" for g in USER_ACTIVITY_GROUPS
    )

summary_value_cols = [
    "s_mat_runtime", "model_runtime", "total_runtime",
    *ordered_ndcg_cols,
]
summary_groups = ["dataset_name", "model"]
df_seed_summary = (
    df_best_results.groupby(summary_groups, sort=True)[summary_value_cols]
    .agg(["mean", "std"])
    .reset_index()
)
df_seed_summary.columns = [
    column[0] if not column[1] else f"{column[0]}_{column[1]}"
    for column in df_seed_summary.columns
]
seed_counts = (
    df_best_results.groupby(summary_groups, sort=True)["seed"]
    .nunique()
    .reset_index(name="n_seeds")
)
df_seed_summary = seed_counts.merge(
    df_seed_summary, on=summary_groups, how="left", validate="one_to_one"
)

frozen_best = best_params_by_tuning_seed[int(TUNING_SEED)]
best_hyperparameter_rows = []
for dataset_cfg in dataset_configs:
    dataset_name = dataset_cfg["name"]
    for model_name, param_key in MODEL_PARAM_KEY.items():
        row = {
            "tuning_seed": int(TUNING_SEED),
            "dataset_name": dataset_name,
            "model": model_name,
        }
        if param_key is None:
            row["validation_selection_score"] = float("nan")
        else:
            tracker = frozen_best[(model_name, dataset_name)]
            row["validation_selection_score"] = tracker["best_score"]
            if tracker["best_params"] is not None:
                row.update(tracker["best_params"])
        best_hyperparameter_rows.append(row)
df_best_hyperparameters = pd.DataFrame(best_hyperparameter_rows)
df_best_hyperparameters.sort_values(
    by=["dataset_name", "model"], inplace=True
)

In [7]:
# Present the single tuning-seed test run (no seed averaging).
print("Test Results (TUNING_SEED)")
tuning_seed_result_metrics = [
    f"ndcg_all@{SELECTION_K}",
    *(f"ndcg_user_{group}@{SELECTION_K}" for group in USER_ACTIVITY_GROUPS),
]
df_tuning_seed_results = df_best_results[
    df_best_results["seed"] == int(TUNING_SEED)
]
display(
    df_tuning_seed_results[["dataset_name", "model", *tuning_seed_result_metrics]]
)

Test Results (TUNING_SEED)


,dataset_name,model,ndcg_all@10,ndcg_user_interaction_0@10,ndcg_user_interaction_1@10,ndcg_user_interaction_2@10,ndcg_user_interaction_3_plus@10
0,01_amz_beauty,01 ItemPop,0.017333,0.105978,0.015152,0.000000,0.000612
1,01_amz_beauty,02 Standard-WMF,0.040186,0.037634,0.062699,0.045461,0.037869
2,01_amz_beauty,03 CoFactor,0.040054,0.037634,0.086703,0.059816,0.033534
3,01_amz_beauty,04 RME,0.047553,0.037634,0.111556,0.072416,0.040311
4,01_amz_beauty,05 NeuMF,0.022227,0.006047,0.025830,0.024565,0.025067
5,01_amz_beauty,06 LightGCN,0.047217,0.045328,0.096520,0.067604,0.040245
6,01_amz_beauty,07 CPARMS-LD,0.053472,0.102004,0.081549,0.052969,0.040324


In [8]:
# Present the frozen best hyperparameters from the tuning seed.
print("Best Hyperparameters")
display(df_best_hyperparameters)

Best Hyperparameters


,tuning_seed,dataset_name,model,validation_selection_score,latent,lambda_rate,n_sweeps,alpha,random_state,lambda_context_rate,...,hidden_layers,n_layers,gamma_like,gamma_dislike,k_user,K_item,min_support,min_confidence,min_lift,normalize
0,42,01_amz_beauty,01 ItemPop,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,42,01_amz_beauty,02 Standard-WMF,0.048297,100.0,1.0000,20.0,5.0,42.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,42,01_amz_beauty,03 CoFactor,0.054515,40.0,0.1000,5.0,40.0,42.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,42,01_amz_beauty,04 RME,0.059793,40.0,0.0001,20.0,20.0,42.0,0.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,42,01_amz_beauty,05 NeuMF,0.027568,32.0,NaN,NaN,NaN,42.0,NaN,...,"(256, 128, 64, 32)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,42,01_amz_beauty,06 LightGCN,0.053250,64.0,0.0001,NaN,NaN,42.0,NaN,...,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,42,01_amz_beauty,07 CPARMS-LD,0.074068,100.0,0.0010,5.0,40.0,42.0,NaN,...,NaN,NaN,0.5,0.0,5.0,2.0,0.001,0.001,1.0,log_row_max


In [9]:
# Present the per-user paired test from the designated statistical-test seed.
df_significance = pd.DataFrame(significance_rows)
if not df_significance.empty:
    df_significance.sort_values(
        by=["dataset_name", "group", "baseline"], inplace=True
    )
    df_significance["sig_90"] = df_significance["p_value"] < 0.10
    df_significance["sig_95"] = df_significance["p_value"] < 0.05
    df_significance["sig_99"] = df_significance["p_value"] < 0.01

print("Statistical Tests")
display(df_significance)

Statistical Tests


,baseline,group,k,n,mean_diff,t_stat,p_value,seed,dataset_name,primary_model,sig_90,sig_95,sig_99
0,01 ItemPop,all,10,968,0.036138,5.280290,1.593110e-07,42,01_amz_beauty,07 CPARMS-LD,True,True,True
5,02 Standard-WMF,all,10,968,0.013286,3.616323,3.142584e-04,42,01_amz_beauty,07 CPARMS-LD,True,True,True
10,03 CoFactor,all,10,968,0.013417,3.372804,7.735797e-04,42,01_amz_beauty,07 CPARMS-LD,True,True,True
15,04 RME,all,10,968,0.005918,1.454364,1.461698e-01,42,01_amz_beauty,07 CPARMS-LD,False,False,False
20,05 NeuMF,all,10,968,0.031245,5.656968,2.028647e-08,42,01_amz_beauty,07 CPARMS-LD,True,True,True
25,06 LightGCN,all,10,968,0.006254,1.390003,1.648479e-01,42,01_amz_beauty,07 CPARMS-LD,False,False,False
1,01 ItemPop,interaction_0,10,145,-0.003974,-0.122835,9.024090e-01,42,01_amz_beauty,07 CPARMS-LD,False,False,False
6,02 Standard-WMF,interaction_0,10,145,0.064370,3.627172,3.969874e-04,42,01_amz_beauty,07 CPARMS-LD,True,True,True
11,03 CoFactor,interaction_0,10,145,0.064370,3.627172,3.969874e-04,42,01_amz_beauty,07 CPARMS-LD,True,True,True
16,04 RME,interaction_0,10,145,0.064370,3.627172,3.969874e-04,42,01_amz_beauty,07 CPARMS-LD,True,True,True


In [10]:
# Present the seed-sensitivity summary (mean/std) for the primary metric.
print("Best Results (mean / std across SENSITIVITY_SEEDS)")
best_result_metrics = [
    f"ndcg_all@{SELECTION_K}",
    *(f"ndcg_user_{group}@{SELECTION_K}" for group in USER_ACTIVITY_GROUPS),
]
best_result_columns = ["dataset_name", "model", "n_seeds"]
for metric in best_result_metrics:
    best_result_columns.extend([f"{metric}_mean", f"{metric}_std"])
display(df_seed_summary[best_result_columns])

Best Results (mean / std across SENSITIVITY_SEEDS)


,dataset_name,model,n_seeds,ndcg_all@10_mean,ndcg_all@10_std,ndcg_user_interaction_0@10_mean,ndcg_user_interaction_0@10_std,ndcg_user_interaction_1@10_mean,ndcg_user_interaction_1@10_std,ndcg_user_interaction_2@10_mean,ndcg_user_interaction_2@10_std,ndcg_user_interaction_3_plus@10_mean,ndcg_user_interaction_3_plus@10_std
0,01_amz_beauty,01 ItemPop,5,0.017333,0.000000,0.105978,0.000000,0.015152,0.000000,0.000000,0.000000,0.000612,0.000000
1,01_amz_beauty,02 Standard-WMF,5,0.040855,0.000818,0.037634,0.000000,0.060509,0.001514,0.047422,0.003646,0.038801,0.000794
2,01_amz_beauty,03 CoFactor,5,0.039547,0.002147,0.037634,0.000000,0.079306,0.006278,0.054476,0.005880,0.034197,0.002624
3,01_amz_beauty,04 RME,5,0.047736,0.000888,0.037634,0.000000,0.106793,0.008661,0.072972,0.005227,0.040972,0.001143
4,01_amz_beauty,05 NeuMF,5,0.018440,0.002848,0.002897,0.002204,0.039382,0.016781,0.019132,0.003792,0.019648,0.003568
5,01_amz_beauty,06 LightGCN,5,0.043708,0.003421,0.028383,0.011682,0.088225,0.017314,0.065332,0.002412,0.039945,0.001655
6,01_amz_beauty,07 CPARMS-LD,5,0.053730,0.000612,0.103715,0.008215,0.074816,0.008946,0.058291,0.004636,0.040323,0.001229


# 7. Save Outputs

In [11]:
# Export frozen hyperparameters, per-seed metrics, sensitivity, and tests.
timestamp_str = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
file_name = f'{RESULTS_DIR}/final_results_{timestamp_str}.xlsx'

ordered_ndcg_cols = []
for k in METRIC_KS:
    ordered_ndcg_cols.append(f"ndcg_all@{int(k)}")
    ordered_ndcg_cols.extend(f"ndcg_user_{g}@{int(k)}" for g in USER_ACTIVITY_GROUPS)

other_cols = [c for c in df_best_results.columns if c not in ordered_ndcg_cols]
df_results = df_best_results[other_cols + ordered_ndcg_cols]

df_dataset_eda = pd.DataFrame(
    [
        {"dataset_name": dataset_name, "split": split_name, **metrics}
        for dataset_name, split_eda in dataset_eda_all.items()
        for split_name, metrics in split_eda.items()
    ]
)

with pd.ExcelWriter(file_name) as writer:
    df_results.to_excel(writer, sheet_name='results', index=False)
    df_seed_summary.to_excel(writer, sheet_name='seed_summary', index=False)
    df_best_hyperparameters.to_excel(
        writer, sheet_name='best_hyperparameters', index=False
    )
    df_significance.to_excel(writer, sheet_name='significance', index=False)
    df_dataset_eda.to_excel(writer, sheet_name='dataset_eda', index=False)
print(f"Saved: {file_name}")

end_time = time.time()
print(f"Elapsed: {end_time - start_time:.1f}s")

Saved: ./results/final_results_20260829_173233.xlsx
Elapsed: 1709.2s
